<a href="https://colab.research.google.com/github/GMISSAGLIA/GM_PyLab/blob/Main/Download_Economic_Data_Part3_ECB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Download Economic & Financial Data from Institutional Data Providers
###   **- Part 3 - ECB - European Central Bank -**

If you don’t have a paid data provider and you’re looking for core economic statistics, you can rely on institutional sources to build your repository, including:

- [Bank for International Settlements -BIS- ](https://stats.bis.org/api-doc/v2/)

- [Federal Reserve Bank of St. Louis -FRED-](https://fred.stlouisfed.org/)   - ⚠️ You'll need a free FRED API key that you can get here: [FRED API](https://fred.stlouisfed.org/docs/api/api_key.html)

- [European Central Bank -ECB-](https://data.ecb.europa.eu/services/site-directory)

- [EUROSTAT](https://ec.europa.eu/eurostat)

- [Banca d'Italia -BANKIT-](https://a2a.bancaditalia.it/infostat/dataservices/)

In this notebook I show how to retrieve data from the  [European Central Bank -ECB-](https://data.ecb.europa.eu/services/site-directory)

I will use SDMX RESTful API web service that offers programmatic access to data and metadata . It provides a standardised interface for interacting with software systems implementing the SDMX standard.[SDMX standard:https://github.com/sdmx-twg/sdmx-rest/](https://github.com/sdmx-twg/sdmx-rest/)

Refer to the official [ECB SDMX RESTful API Guide](https://data.ecb.europa.eu/help/api/overview) for full specifications and  to the
[guide to download ECB data:](https://data.ecb.europa.eu/help/api/data)

I demonstrate how to use the API and implement a set of utility functions to interact with the Statistical Database, along with several examples.

In [ ]:
###########################################################################
# we have to install versions that don't give compatibility problems
%pip install -U "sdmx1>=2.12" "pydantic>=1.10.21"
###########################################################################
import subprocess
import sys
# If needed, install dependencies (uncomment to run)
def install_packages():
    packages = [
        'pandasql', 'matplotlib', 'seaborn', 'scikit-learn',
        'statsmodels', 'scipy', 'requests', 'openpyxl','pyarrow', 'yfinance',
        'eurostat', 'ecbdata', 'fredapi', 'jsonstat.py', 'beautifulsoup4', 'lxml']
    for package in packages:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        except:
            print(f"Package {package} already installed or failed to install")
install_packages()
##############################################################################
#Standard Library
import math
import gzip
import zipfile
import io
import os
import shutil
import subprocess
import sys
import time
from datetime import datetime, timedelta
import warnings
from functools import reduce
from io import StringIO
import tkinter as tk
from tkinter import Tk, filedialog
from tkinter.filedialog import askopenfilename
from IPython import display
from IPython.display import display # Import the standard display function

# Other Library Imports
import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pandas_datareader.data as web
from pandasql import sqldf
import requests
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.api import VAR, VECM
from statsmodels.tsa.stattools import adfuller, coint
from pathlib import PurePosixPath
from bs4 import BeautifulSoup
from urllib.parse import urlparse

from fredapi import Fred
import eurostat
from ecbdata import ecbdata

# Define the SQL runner
pysqldf_G = lambda q: sqldf(q, globals())
pysqldf_L = lambda q: sqldf(q, locals())
warnings.filterwarnings("ignore")

##############################################################################
#Constants
EUROSTAT3_URL = "https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/data"
EUROSTAT2_URL = "https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data"
ECB_SDW = "https://sdw-wsrest.ecb.europa.eu/service/data"
ECB_BULK_DATA ="https://data-api.ecb.europa.eu/service/data"
BANKIT_URL = "https://a2a.bancaditalia.it/infostat/dataservices/export"
DT_START ='2001-12-31'
DT_END = '2024-12-31'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.6/244.6 kB 4.1 MB/s eta 0:00:00
Package jsonstat.py already installed or failed to install


In [ ]:
##############################################################################
# Step 1:  the "hello world" of ECB API queries - retrieving one time series
# in raw JSON form - the most basic example of how to query the ECB Data API.
# - It builds a simple URL request pointing to one specific BSI series
#  (Monetary Financial Institutions, Italy).
# - No filtering parameters, transformations, or adjustments are applied:
#  the request just asks the API for the raw series data.
# - The request includes a header specifying that we want the response
#  in SDMX-JSON format (the standard JSON encoding used by ECB/SDMX).
# - The response is then parsed into a Python dictionary with .json().

url = "https://data-api.ecb.europa.eu/service/data/BSI/M.IT.N.A.A20.A.1.U2.2240.Z01.E"
response = requests.get(url, headers={"Accept": "application/vnd.sdmx.data+json"})
data = response.json()
display(data)

{'header': {'id': 'd4402082-5fb6-46e6-874f-82cbf55dd113',
  'test': False,
  'prepared': '2025-09-28T09:02:38.908+00:00',
  'sender': {'id': 'ECB'}},
 'dataSets': [{'action': 'Replace',
   'validFrom': '2025-09-28T09:02:38.908+00:00',
   'series': {'0:0:0:0:0:0:0:0:0:0:0': {'attributes': [0,
      None,
      0,
      None,
      None,
      None,
      None,
      None,
      None,
      None,
      None,
      0,
      None,
      None,
      0,
      0,
      0,
      0],
     'observations': {'0': [550083, 0, 0, None, None],
      '1': [552340, 0, 0, None, None],
      '2': [551633, 0, 0, None, None],
      '3': [552928, 0, 0, None, None],
      '4': [552368, 0, 0, None, None],
      '5': [561466, 0, 0, None, None],
      '6': [563512, 0, 0, None, None],
      '7': [565362, 0, 0, None, None],
      '8': [569741, 0, 0, None, None],
      '9': [570057, 0, 0, None, None],
      '10': [572426, 0, 0, None, None],
      '11': [588676, 0, 0, None, None],
      '12': [588051, 0, 0, None, N

In [ ]:
flow = "YC"  # e.g., yield curve dataset
params = {"detail": "serieskeysonly", "format": "csvdata"}  # keys only, SDMX-CSV
r = requests.get(f"https://data-api.ecb.europa.eu/service/data/{flow}",
                 params=params, timeout=600)
r.raise_for_status()
with open(f"{flow}_serieskeys.csv","wb") as f:
    f.write(r.content)  # open in pandas later

# Quick read
import pandas as pd
keys = pd.read_csv(f"{flow}_serieskeys.csv")
keys.head()


,KEY,FREQ,REF_AREA,CURRENCY,PROVIDER_FM,INSTRUMENT_FM,PROVIDER_FM_ID,DATA_TYPE_FM
0,YC.B.U2.EUR.4F.G_N_C.SV_C_YM.SR_27Y8M,B,U2,EUR,4F,G_N_C,SV_C_YM,SR_27Y8M
1,YC.B.U2.EUR.4F.G_N_C.SV_C_YM.SR_28Y11M,B,U2,EUR,4F,G_N_C,SV_C_YM,SR_28Y11M
2,YC.B.U2.EUR.4F.G_N_C.SV_C_YM.SR_4Y1M,B,U2,EUR,4F,G_N_C,SV_C_YM,SR_4Y1M
3,YC.B.U2.EUR.4F.G_N_C.SV_C_YM.SR_4Y2M,B,U2,EUR,4F,G_N_C,SV_C_YM,SR_4Y2M
4,YC.B.U2.EUR.4F.G_N_C.SV_C_YM.SR_5Y7M,B,U2,EUR,4F,G_N_C,SV_C_YM,SR_5Y7M


In [ ]:
# Step 2: Requesting a single series from the ECB Data API
# This function wraps the raw request logic into something reusable,
# adding small enhancements for easier use.
def get_ecb_series(flow: str, key: str, timeout=120):
    """
    Download a single ECB data series from the ECB Data API in CSV format
    and return it as a pandas DataFrame.
    Parameters
    ----------
    flow : str
        The dataset (dataflow) code, e.g. "RDF" for Risk Dashboard,
        "EXR" for exchange rates, "MNA" for macroeconomic accounts, etc.
    key : str
        The series key identifying the specific dimension combination,
        e.g. "Q.IT.EUR.4F.CR.DCGDPG.RO" for Italy domestic credit-to-GDP gap.
    timeout : int, optional
        Maximum time (in seconds) allowed for the request before failing.

    Returns
    -------
    pandas.DataFrame
        DataFrame containing the requested series in tidy tabular form,
        with OBS_VALUE already converted to numeric.
    """

    BASE = "https://data-api.ecb.europa.eu/service/data"

    # Build the full request URL:
    # /service/data/{flow}/{key}?format=csvdata
    #   → flow selects the dataset
    #   → key selects the exact series within the dataset
    #   → format=csvdata requests CSV for easy pandas ingestion
    url = f"{BASE}/{flow}/{key}?format=csvdata"

    # Read CSV directly into a DataFrame.
    # dtype=str keeps all columns as strings initially (safe, avoids auto-parsing issues).
    df = pd.read_csv(url, dtype=str)

    # Convert the OBS_VALUE column (the actual numeric observations) to float.
    # Any parsing errors (e.g. missing values, dots, non-numeric strings) become NaN.
    df["OBS_VALUE"] = pd.to_numeric(df["OBS_VALUE"], errors="coerce")

    return df

###############################################################################
# Example usage: retrieve Italy's domestic credit-to-GDP gap from the ECB Risk Dashboard
flow = "RDF"
key  = "Q.IT.EUR.4F.CR.DCGDPG.RO"

# Call the function with flow and key
df = get_ecb_series(flow, key)

# Preview the first rows of the series
display(df.head())

# Print a visual separator
print('#' * 100)

# Show the column names available in the dataset
display(df.columns)


,KEY,FREQ,REF_AREA,CURRENCY,PROVIDER_FM,INSTRUMENT_FM,PROVIDER_FM_ID,DATA_TYPE_FM,TIME_PERIOD,OBS_VALUE,...,UNIT_INDEX_BASE,COMPILATION,COVERAGE,DECIMALS,SOURCE_AGENCY,SOURCE_PUB,TITLE,TITLE_COMPL,UNIT,UNIT_MULT
0,RDF.Q.IT.EUR.4F.CR.DCGDPG.RO,Q,IT,EUR,4F,CR,DCGDPG,RO,1997-Q4,0.82,...,NaN,NaN,NaN,2,4F0,NaN,NaN,Italy,PC,0
1,RDF.Q.IT.EUR.4F.CR.DCGDPG.RO,Q,IT,EUR,4F,CR,DCGDPG,RO,1998-Q1,-0.13,...,NaN,NaN,NaN,2,4F0,NaN,NaN,Italy,PC,0
2,RDF.Q.IT.EUR.4F.CR.DCGDPG.RO,Q,IT,EUR,4F,CR,DCGDPG,RO,1998-Q2,0.10,...,NaN,NaN,NaN,2,4F0,NaN,NaN,Italy,PC,0
3,RDF.Q.IT.EUR.4F.CR.DCGDPG.RO,Q,IT,EUR,4F,CR,DCGDPG,RO,1998-Q3,-1.16,...,NaN,NaN,NaN,2,4F0,NaN,NaN,Italy,PC,0
4,RDF.Q.IT.EUR.4F.CR.DCGDPG.RO,Q,IT,EUR,4F,CR,DCGDPG,RO,1998-Q4,1.47,...,NaN,NaN,NaN,2,4F0,NaN,NaN,Italy,PC,0


####################################################################################################


Index(['KEY', 'FREQ', 'REF_AREA', 'CURRENCY', 'PROVIDER_FM', 'INSTRUMENT_FM',
       'PROVIDER_FM_ID', 'DATA_TYPE_FM', 'TIME_PERIOD', 'OBS_VALUE',
       'OBS_STATUS', 'OBS_CONF', 'OBS_PRE_BREAK', 'OBS_COM', 'TIME_FORMAT',
       'BREAKS', 'COLLECTION', 'COMPILING_ORG', 'DISS_ORG', 'DOM_SER_IDS',
       'FM_CONTRACT_TIME', 'FM_COUPON_RATE', 'FM_IDENTIFIER', 'FM_LOT_SIZE',
       'FM_MATURITY', 'FM_OUTS_AMOUNT', 'FM_PUT_CALL', 'FM_STRIKE_PRICE',
       'PUBL_MU', 'PUBL_PUBLIC', 'UNIT_INDEX_BASE', 'COMPILATION', 'COVERAGE',
       'DECIMALS', 'SOURCE_AGENCY', 'SOURCE_PUB', 'TITLE', 'TITLE_COMPL',
       'UNIT', 'UNIT_MULT'],
      dtype='object')

In [ ]:
# Import tools for robust HTTP requests with retry logic
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def make_session(retries=3, backoff=0.5, status_forcelist=(429, 500, 502, 503, 504)):
    """
    Create and configure a requests.Session object with built-in retry logic.
    Parameters
    ----------
    retries : int
        Maximum number of total retry attempts for failed requests.
        Applies to connection errors, read errors, and specific HTTP status codes.
    backoff : float
        Backoff factor (in seconds). Determines the delay between retries:
        delay = backoff * (2 ** (retry_number - 1)).
        Example: with backoff=0.5, retries happen after 0.5s, 1s, 2s...
    status_forcelist : tuple of int
        HTTP status codes that should trigger a retry.
        Defaults include: 429 (Too Many Requests), 500, 502, 503, 504.

    Returns
    -------
    requests.Session
        A session object that automatically retries failed GET requests
        using the defined policy.
    """

    # Start a new persistent session (shares connections, cookies, headers)
    s = requests.Session()

    # Configure retry behavior
    retry = Retry(
        total=retries,              # max number of retries in total
        read=retries,               # retries on read errors
        connect=retries,            # retries on connection errors
        backoff_factor=backoff,     # exponential backoff factor for delays
        status_forcelist=status_forcelist,  # which HTTP status codes trigger retry
        allowed_methods=frozenset(["GET"]), # only retry GET requests (not POST/PUT/DELETE)
        raise_on_status=False       # do not raise exception immediately on bad status
    )

    # Mount the retry policy onto both HTTP and HTTPS connections
    s.mount("https://", HTTPAdapter(max_retries=retry))
    s.mount("http://", HTTPAdapter(max_retries=retry))

    return s

# Create a global session object that can be reused across requests.
# This saves overhead (persistent connections) and applies retry logic everywhere.
SESSION = make_session()

# Define a global default timeout (in seconds) for API requests.
# Ensures requests don't hang indefinitely if the server is slow/unresponsive.
DEFAULT_TIMEOUT = 180


In [ ]:
# pip install requests beautifulsoup4 lxml pandas
# import requests, pandas as pd
# from bs4 import BeautifulSoup
# from urllib.parse import urlparse

"""
Scrape the ECB "Datasets" web page to retrieve the list of available dataset codes
that can be queried through the ECB Data API.

Why scrape?
- The ECB Data Portal lists datasets at https://data.ecb.europa.eu/data/datasets
- Each dataset has a unique "flow" code (e.g. EXR for exchange rates, MNA for macroeconomic accounts)
- This function automates collecting those codes so the user can later query them programmatically.
"""
def ecb_list_dataset_codes(
    sess: requests.session = SESSION,
    url: str = "https://data.ecb.europa.eu/data/datasets",
    timeout: int = 120,
    headers: dict | None = None,
    verbose: bool = False
) -> pd.DataFrame:

    # -----------------------------------------------------------
    # 1. Prepare HTTP headers
    # ------------------------------------------------------------
    # Many websites block "bare" requests, so we spoof a real browser User-Agent
    # and set Accept-Language. If the caller provides custom headers, use them.
    if headers is None:
        headers = {
            "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                           "AppleWebKit/537.36 (KHTML, like Gecko) "
                           "Chrome/124.0 Safari/537.36"),
            "Accept-Language": "en;q=0.9,it;q=0.8",
        }

    # ------------------------------------------------------------------------
    # 2. Build or reuse a session
    # ------------------------------------------------------------------------
    # If the caller did not provide a session, create one with retry logic.
    sess = make_session() if sess is None else sess

    #------------------------------------------------------------
    # 3. Download the datasets page and parse HTML
    #------------------------------------------------------------
    html = sess.get(url, headers=headers, timeout=timeout).text
    soup = BeautifulSoup(html, "lxml")

    #-------------------------------------------------------------
    # 4. Look for anchor tags containing ECB Data API links
    #-------------------------------------------------------------
    # ECB dataset links look like:
    #   https://data-api.ecb.europa.eu/service/data/{DATASET_CODE}
    # So we filter <a> tags whose href contains both "data-api.ecb.europa.eu"
    # and "/service/data/".
    rows = []
    for a in soup.select('a[href*="data-api.ecb.europa.eu"][href*="/service/data/"]'):
        href = a.get("href", "")
        if not href:
            continue

        # Extract dataset code = last part of the path
        code = urlparse(href).path.rstrip("/").split("/")[-1]
        if not code:
            continue

        # Store dataset code into our list
        if code:
            row = {"ID": code, "DataSetCode": code}
            rows.append(row)
        elif verbose:
            # In verbose mode, warn if we find a link without a usable code
            print(f"[warn] anchor without valid dataset code: {href!r}")

    #----------------------------------------------------
    # 5. Build tidy DataFrame
    #----------------------------------------------------------
    df = pd.DataFrame(rows).drop_duplicates(subset=["ID"]).sort_values("ID")

    # If no datasets found, return empty DataFrame
    if df.empty:
        return df

    # Otherwise, set ID as index for convenience
    df.set_index("ID", inplace=True)
    return df


################################################################
# Example usage: scrape the dataset page and list all ECB dataset codes
df_ecb_flows = ecb_list_dataset_codes()
df_ecb_flows


,DataSetCode
ID,
AGR,AGR
AME,AME
BKN,BKN
BLS,BLS
BNT,BNT
...,...
SUR,SUR
TGB,TGB
TRD,TRD


In [ ]:
# pip install requests beautifulsoup4 lxml pandas
# import requests, pandas as pd
# from bs4 import BeautifulSoup
# from urllib.parse import urlparse

def get_ecb_dataset_catalog(
    sess: requests.session = SESSION,
    base: str = "https://data.ecb.europa.eu/data/datasets",
    headers: dict | None = None,
    timeout_list: int = 120,
    timeout_ds: int = 60,
    drop_discontinued: bool = True,
    set_index: bool = True,
    verbose: bool = False,
) -> pd.DataFrame:
    """
    Scrape the ECB 'Datasets' portal and build a catalog of available datasets.

    The ECB Data Portal lists all datasets (flows). Each dataset has:
      - a machine code (DataSetCode), e.g. "EXR", "MNA", "YC"
      - a human-readable description, e.g. "Exchange Rates", "Macroeconomic accounts"

    This function:
      1. Downloads the ECB dataset list page
      2. Extracts dataset codes from links to the ECB API
      3. Visits each dataset’s detail page to parse the <h1> title and extract
         a human-readable description
      4. Filters out invalid or discontinued entries
      5. Returns a tidy DataFrame mapping codes → descriptions

    Parameters
    ----------
    sess : requests.Session
        A session object, allows persistent connections + retries.
    base : str
        Base URL of the ECB datasets listing page.
    headers : dict
        HTTP headers to use; default mimics a browser User-Agent.
    timeout_list : int
        Timeout (seconds) for the dataset listing page request.
    timeout_ds : int
        Timeout (seconds) for each dataset detail page request.
    drop_discontinued : bool
        If True, skip datasets marked "discontinued" in their title.
    set_index : bool
        If True, set the DataFrame index to the dataset code.
    verbose : bool
        If True, print warnings for codes that cannot be parsed.

    Returns
    -------
    pandas.DataFrame
        Columns: DataSetCode, DataSetDes
        Index: DataSetCode (if set_index=True)
    """

    #------------------------------------------------------------
    # 1) HTTP headers: fake a browser User-Agent so the ECB site accepts the request
    #-----------------------------------------------------------
    if headers is None:
        headers = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/124.0 Safari/537.36"
            ),
            "Accept-Language": "en;q=0.9,it;q=0.8",
        }

    # Ensure we have a session (persistent HTTP client with retry logic)
    sess = make_session() if sess is None else sess

    #--------------------------------------------------------    # 2) Fetch the datasets overview page and parse with BeautifulSoup
    # -----------------------------------------------------------
    resp = sess.get(base, headers=headers, timeout=timeout_list)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "lxml")

    # Collect dataset codes from links to the API
    codes = sorted({
        urlparse(a["href"]).path.rstrip("/").split("/")[-1]
        for a in soup.select('a[href*="data-api.ecb.europa.eu"][href*="/service/data/"]')
        if a.get("href")
    })

    # -------------------------------------------------------------------------
    # 3) Visit each dataset’s own detail page and parse its <h1> title
    # -------------------------------------------------------------------------
    rows = []
    for code in codes:
        try:
            r = sess.get(base + "/" + code, headers=headers, timeout=timeout_ds)
            r.raise_for_status()
            s2 = BeautifulSoup(r.text, "lxml")

            # Example of an H1 title: "Exchange Rates - EXR"
            h1 = s2.find("h1")
            if not h1:
                if verbose:
                    print(f"[warn] no <h1> for {code}")
                continue

            title = h1.get_text(" ", strip=True)
            sep = f" - {code}"
            # Extract the part before " - CODE"
            desc = title.split(sep)[0].strip() if sep in title else title.strip()

            # -----------------------------------------------------------------
            # 4) Apply filters: skip invalid or discontinued entries
            # -----------------------------------------------------------------
            if not desc or desc.upper() == code.upper():
                continue
            if drop_discontinued and "discontinued" in desc.lower():
                continue

            rows.append({"DataSetCode": code, "DataSetDes": desc})

        except Exception as e:
            if verbose:
                print(f"[warn] skip {code}: {e}")

    # -------------------------------------------------------------------------
    # 5) Build final DataFrame
    # -------------------------------------------------------------------------
    df = (pd.DataFrame(rows)
            .drop_duplicates("DataSetCode")
            .sort_values("DataSetCode")
            .reset_index(drop=True))

    if set_index and not df.empty:
        df["ID"] = df["DataSetCode"]
        df.set_index("ID", inplace=True)

    return df


##############################################################################
# Example usage:
df_ecb = get_ecb_dataset_catalog(verbose=False)
display(df_ecb)


,DataSetCode,DataSetDes
ID,,
AGR,AGR,Farm-gate and wholesale market prices
AME,AME,AMECO
BKN,BKN,Banknotes statistics
BLS,BLS,Bank Lending Survey Statistics
BNT,BNT,Shipments of Euro Banknotes Statistics
...,...,...
SUR,SUR,Opinion Surveys
TGB,TGB,Target Balances
TRD,TRD,Eurostat External Trade Statistics


In [ ]:
# pip install requests beautifulsoup4 lxml pandas
# import time, re
# import requests, pandas as pd
# from bs4 import BeautifulSoup
# from urllib.parse import urlparse

def get_ecb_dataset_catalog_rich(
    sess: requests.Session = SESSION,
    base_list: str = "https://data.ecb.europa.eu/data/datasets",
    base_ds: str   = "https://data.ecb.europa.eu/data/datasets/{code}",
    headers: dict | None = None,
    timeout_list: int = 120,
    timeout_ds: int = 60,
    drop_discontinued: bool = True,
    set_index: bool = True,
    sleep_between: float = 0.3,   # polite throttling between page hits
    verbose: bool = False,
) -> pd.DataFrame:
    """
    Build a catalog of ECB datasets with:
      - DataSetCode (machine code, e.g., EXR)
      - DataSetDes  (title from the <h1>, e.g., 'Exchange rates')
      - DataSetInfo (short description text from the page body)

    How it works
    ------------
    1) Scrape the datasets listing page to discover dataset codes.
    2) For each code, open the dataset page and parse:
       - <h1> to get a human-readable title (DataSetDes)
       - a short description block (DataSetInfo), using robust CSS fallbacks
    3) Filter out empty/duplicate/“discontinued” entries (optional).

    Notes
    -----
    - The ECB site can evolve; selectors use multiple fallbacks to stay resilient.
    - `sleep_between` avoids hammering the site (be a good citizen).
    """

    # -- Headers (pretend to be a normal browser) -----------------------------
    if headers is None:
        headers = {
            "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                           "AppleWebKit/537.36 (KHTML, like Gecko) "
                           "Chrome/124.0 Safari/537.36"),
            "Accept-Language": "en;q=0.9,it;q=0.8",
        }

    # -- Session with retries (yours) -----------------------------------------
    sess = make_session() if sess is None else sess

    # -- Step 1: get dataset codes from the listing page ----------------------
    resp = sess.get(base_list, headers=headers, timeout=timeout_list)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "lxml")

    codes = sorted({
        urlparse(a["href"]).path.rstrip("/").split("/")[-1]
        for a in soup.select('a[href*="data-api.ecb.europa.eu"][href*="/service/data/"]')
        if a.get("href")
    })

    if verbose:
        print(f"Found {len(codes)} dataset codes")

    # -- Helper: extract the best short description text from a dataset page --
    def _extract_description(soup_ds: BeautifulSoup) -> str:
        # Primary candidates: page intro/summary sections commonly used on the portal
        candidates = [
            ".dataset-intro", ".intro", ".content-intro",
            ".ds-header__intro", ".ds-intro", ".hero__intro",
        ]
        for sel in candidates:
            node = soup_ds.select_one(sel)
            if node:
                txt = " ".join(node.get_text(" ", strip=True).split())
                if len(txt) > 30:
                    return txt

        # Fallback 1: first non-empty paragraph after the H1
        h1 = soup_ds.find("h1")
        if h1:
            for p in h1.find_all_next(["p", "div"], limit=10):
                txt = " ".join(p.get_text(" ", strip=True).split())
                if len(txt) > 30:
                    return txt

        # Fallback 2: first meaningful paragraph on the page
        for p in soup_ds.find_all("p"):
            txt = " ".join(p.get_text(" ", strip=True).split())
            if len(txt) > 30:
                return txt

        return ""  # nothing decent found

    # -- Step 2: open each dataset page, parse <h1> and a short description ---
    rows = []
    for code in codes:
        try:
            url_ds = base_ds.format(code=code)
            r = sess.get(url_ds, headers=headers, timeout=timeout_ds)
            r.raise_for_status()
            s2 = BeautifulSoup(r.text, "lxml")

            # Title in <h1>, often like "Exchange rates - EXR"
            h1 = s2.find("h1")
            if not h1:
                if verbose:
                    print(f"[warn] no <h1> for {code}")
                time.sleep(sleep_between)
                continue

            title = h1.get_text(" ", strip=True)
            sep = f" - {code}"
            desc = title.split(sep)[0].strip() if sep in title else title.strip()

            # Short description block
            info = _extract_description(s2)

            # Filters
            if not desc or desc.upper() == code.upper():
                if verbose:
                    print(f"[skip] empty/identical title for {code}")
                time.sleep(sleep_between)
                continue
            if drop_discontinued and "discontinued" in desc.lower():
                if verbose:
                    print(f"[skip] discontinued {code}")
                time.sleep(sleep_between)
                continue

            rows.append({
                "DataSetCode": code,
                "DataSetDes":  desc,
                "DataSetInfo": info
            })

        except Exception as e:
            if verbose:
                print(f"[warn] {code}: {e}")

        # Polite delay between requests
        time.sleep(sleep_between)

    # -- Build final DataFrame -------------------------------------------------
    df = (pd.DataFrame(rows)
            .drop_duplicates("DataSetCode")
            .sort_values("DataSetCode")
            .reset_index(drop=True))

    if set_index and not df.empty:
        df["ID"] = df["DataSetCode"]
        df.set_index("ID", inplace=True)

    return df

# Example
df_ecb = get_ecb_dataset_catalog_rich(verbose=False)
display(df_ecb.head())


,DataSetCode,DataSetDes,DataSetInfo
ID,,,
AGR,AGR,Farm-gate and wholesale market prices,Data (10) Data information Data structure Sele...
AME,AME,AMECO,Data (121) Data information Data structure Sel...
BKN,BKN,Banknotes statistics,Data (824) Data information Data structure Pub...
BLS,BLS,Bank Lending Survey Statistics,Data (21753) Data information Data structure P...
BNT,BNT,Shipments of Euro Banknotes Statistics,Data (4) Data information Data structure Selec...


In [ ]:
"""
Bulk Data Download: fetch all the series inside a given ECB dataset (flow).
For example:
    - dataset_code="YC"  → returns all yield curve series
    - dataset_code="RDF" → returns all credit-to-GDP gap series
"""

def download_ecb_bulk(
    dataset_code: str,
    DataSetDes: str = None,
    sess: requests.session = SESSION,
    startperiod: str | None = None,
    updatedafter: str | None = None,
    timeout: int = 600,
    BASE: str = "https://data-api.ecb.europa.eu/service/data"
) -> pd.DataFrame:

    # ---------------------------------------------------------------------
    # 1) Build request URL and query parameters
    # ---------------------------------------------------------------------
    url = f"{BASE}/{dataset_code}"
    params = {
        "format": "csvdata",   # request CSV format (other options: jsondata, structurespecificdata, genericdata)
        "detail": "dataonly",  # skip structural metadata, keep just data
    }
    # Optional filters: start date or “updated after” timestamp
    if startperiod:
        params["startPeriod"] = startperiod
    if updatedafter:
        params["updatedAfter"] = updatedafter

    # HTTP headers: accept compressed CSV
    headers = {"Accept": "text/csv", "Accept-Encoding": "gzip"}

    # Ensure we have a retry-enabled session
    sess = make_session() if sess is None else sess

    # ---------------------------------------------------------------------
    # 2) Download the bulk dataset to a local CSV file
    # ---------------------------------------------------------------------
    # Stream the response to avoid holding a huge file in memory.
    # Save chunks of 1 MB (1<<20 bytes) to disk as {dataset_code}.csv
    with sess.get(url, params=params, headers=headers, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with open(f"{dataset_code}.csv", "wb") as f:
            for chunk in r.iter_content(1 << 20):
                if chunk:   # filter out keep-alive chunks
                    f.write(chunk)

    # ---------------------------------------------------------------------
    # 3) Load the CSV into a pandas DataFrame
    # ---------------------------------------------------------------------
    df = pd.read_csv(f"{dataset_code}.csv", low_memory=False, dtype=str)

    # Convert the OBS_VALUE column (the numeric observation values) if present
    if "OBS_VALUE" in df.columns:
        df["OBS_VALUE"] = pd.to_numeric(df["OBS_VALUE"], errors="coerce")

    # Add a column with the dataset description (or fallback to dataset_code)
    DataSetDes = dataset_code if DataSetDes is None else DataSetDes
    df["DataSetDes"] = DataSetDes

    # ---------------------------------------------------------------------
    # 4) Reorder columns so 'DataSetDes' appears first (for clarity)
    # ---------------------------------------------------------------------
    cols = df.columns.tolist()
    if "DataSetDes" in cols:
        cols.insert(0, cols.pop(cols.index("DataSetDes")))
        df = df[cols]

    return df
# -------------------------------------------------------------------------
# Example usage
# -------------------------------------------------------------------------
# Example 1: Yield curve dataset (all series)
dataset_code = "YC"
df = download_ecb_bulk(dataset_code)
display(df)
print('#' * 120)
# Show the unique series keys available in this dataset
display(pd.DataFrame(df['KEY'].unique()))
# Example 2: Credit-to-GDP gap dataset
dataset_code = "RDF"
df = download_ecb_bulk(dataset_code)
display(df)
print('#' * 120)
display(pd.DataFrame(df['KEY'].unique()))


,DataSetDes,KEY,FREQ,REF_AREA,CURRENCY,PROVIDER_FM,INSTRUMENT_FM,PROVIDER_FM_ID,DATA_TYPE_FM,TIME_PERIOD,OBS_VALUE
0,YC,YC.B.U2.EUR.4F.G_N_A.SV_C_YM.BETA0,B,U2,EUR,4F,G_N_A,SV_C_YM,BETA0,2004-09-06,5.410510
1,YC,YC.B.U2.EUR.4F.G_N_A.SV_C_YM.BETA0,B,U2,EUR,4F,G_N_A,SV_C_YM,BETA0,2004-09-07,5.391886
2,YC,YC.B.U2.EUR.4F.G_N_A.SV_C_YM.BETA0,B,U2,EUR,4F,G_N_A,SV_C_YM,BETA0,2004-09-08,5.385978
3,YC,YC.B.U2.EUR.4F.G_N_A.SV_C_YM.BETA0,B,U2,EUR,4F,G_N_A,SV_C_YM,BETA0,2004-09-09,5.377333
4,YC,YC.B.U2.EUR.4F.G_N_A.SV_C_YM.BETA0,B,U2,EUR,4F,G_N_A,SV_C_YM,BETA0,2004-09-10,5.355732
...,...,...,...,...,...,...,...,...,...,...,...
11651820,YC,YC.B.U2.EUR.4F.G_N_W.SV_C_YM.SR_6M,B,U2,EUR,4F,G_N_W,SV_C_YM,SR_6M,2025-09-22,1.971187
11651821,YC,YC.B.U2.EUR.4F.G_N_W.SV_C_YM.SR_6M,B,U2,EUR,4F,G_N_W,SV_C_YM,SR_6M,2025-09-23,1.971087
11651822,YC,YC.B.U2.EUR.4F.G_N_W.SV_C_YM.SR_6M,B,U2,EUR,4F,G_N_W,SV_C_YM,SR_6M,2025-09-24,1.977436
11651823,YC,YC.B.U2.EUR.4F.G_N_W.SV_C_YM.SR_6M,B,U2,EUR,4F,G_N_W,SV_C_YM,SR_6M,2025-09-25,1.973274


########################################################################################################################


,0
0,YC.B.U2.EUR.4F.G_N_A.SV_C_YM.BETA0
1,YC.B.U2.EUR.4F.G_N_A.SV_C_YM.BETA1
2,YC.B.U2.EUR.4F.G_N_A.SV_C_YM.BETA2
3,YC.B.U2.EUR.4F.G_N_A.SV_C_YM.BETA3
4,YC.B.U2.EUR.4F.G_N_A.SV_C_YM.IF_10M
...,...
2160,YC.B.U2.EUR.4F.G_N_C.SV_C_YM.SR_9Y9M
2161,YC.B.U2.EUR.4F.G_N_C.SV_C_YM.TAU1
2162,YC.B.U2.EUR.4F.G_N_C.SV_C_YM.TAU2
2163,YC.B.U2.EUR.4F.G_N_W.SV_C_YM.SR_3M


,DataSetDes,KEY,FREQ,REF_AREA,CURRENCY,PROVIDER_FM,INSTRUMENT_FM,PROVIDER_FM_ID,DATA_TYPE_FM,TIME_PERIOD,OBS_VALUE
0,RDF,RDF.D.D0.Z0Z.4F.EC.DFTLB.PR,D,D0,Z0Z,4F,EC,DFTLB,PR,2007-01-01,0.02
1,RDF,RDF.D.D0.Z0Z.4F.EC.DFTLB.PR,D,D0,Z0Z,4F,EC,DFTLB,PR,2007-01-02,0.02
2,RDF,RDF.D.D0.Z0Z.4F.EC.DFTLB.PR,D,D0,Z0Z,4F,EC,DFTLB,PR,2007-01-03,0.02
3,RDF,RDF.D.D0.Z0Z.4F.EC.DFTLB.PR,D,D0,Z0Z,4F,EC,DFTLB,PR,2007-01-04,0.02
4,RDF,RDF.D.D0.Z0Z.4F.EC.DFTLB.PR,D,D0,Z0Z,4F,EC,DFTLB,PR,2007-01-05,0.02
...,...,...,...,...,...,...,...,...,...,...,...
12626,RDF,RDF.Q.U2.EUR.4F.CR.DCGDPG.RO,Q,U2,EUR,4F,CR,DCGDPG,RO,2024-Q1,-19.60
12627,RDF,RDF.Q.U2.EUR.4F.CR.DCGDPG.RO,Q,U2,EUR,4F,CR,DCGDPG,RO,2024-Q2,-19.60
12628,RDF,RDF.Q.U2.EUR.4F.CR.DCGDPG.RO,Q,U2,EUR,4F,CR,DCGDPG,RO,2024-Q3,-20.03
12629,RDF,RDF.Q.U2.EUR.4F.CR.DCGDPG.RO,Q,U2,EUR,4F,CR,DCGDPG,RO,2024-Q4,-19.44


########################################################################################################################


,0
0,RDF.D.D0.Z0Z.4F.EC.DFTLB.PR
1,RDF.D.D0.Z0Z.4F.EC.DFTSV.PR
2,RDF.Q.AT.EUR.4F.CR.DCGDPG.RO
3,RDF.Q.BE.EUR.4F.CR.DCGDPG.RO
4,RDF.Q.BG.EUR.4F.CR.DCGDPG.RO
5,RDF.Q.CY.EUR.4F.CR.DCGDPG.RO
6,RDF.Q.CZ.EUR.4F.CR.DCGDPG.RO
7,RDF.Q.DE.EUR.4F.CR.DCGDPG.RO
8,RDF.Q.DK.EUR.4F.CR.DCGDPG.RO
9,RDF.Q.EE.EUR.4F.CR.DCGDPG.RO


In [ ]:
# -------------------------------------------------------------------------
# Example usage
# Get all the Datasets listed in df_ecb (we create in previous step)
# -------------------------------------------------------------------------
dfs = [
    download_ecb_bulk(dataset_code=code,DataSetDes=des)
    for code, des in df_ecb[["DataSetCode", "DataSetDes"]].itertuples(index=False, name=None)
]
# Create a unique dataframe concatenating all the dataframe
DF_ECB_BULK_ALL = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
display(DF_ECB_BULK_ALL.head())

In [ ]:
#ECB codelist map
def _build_codelist_maps(struct):
    """Crea dizionario {codelist_id: {code_id: code_label}} dalle codelist SDMX."""
    maps = {}
    cl_root = (struct or {}).get("codelists", {}).get("codelist", [])
    for cl in cl_root:
        cl_id = cl.get("id")
        entries = {}
        for code in cl.get("codes", cl.get("code", [])):
            cid = code.get("id")
            # name può essere dizionario con localizzazioni; prendi 'en' o prima disponibile
            name = code.get("name")
            if isinstance(name, dict):
                label = name.get("en") or next(iter(name.values()), cid)
            else:
                label = name or cid
            entries[cid] = label
        if cl_id:
            maps[cl_id] = entries
    return maps

def _label_from_codelist(codelist_maps, codelist_id, code):
    m = codelist_maps.get(codelist_id, {})
    return m.get(code, code)

def ecb_series_with_metadata(flow: str, key: str, params=None, timeout=30):
    """
    Scarica una serie ECB (SDMX-JSON) e restituisce:
    - df: DataFrame con TIME_PERIOD, VALUE e attributi osservazione (anche in versione *_LABEL)
    - series_meta: dict {dimension_id: {"code": ..., "label": ...}} per tutte le dimensioni di serie
    """
    base = "https://data-api.ecb.europa.eu/service/data"
    headers = {"Accept": "application/vnd.sdmx.data+json"}
    params = params or {}

    # 1) fetch
    url = f"{base}/{flow}/{key}"
    r = requests.get(url, headers=headers, params=params, timeout=timeout)
    r.raise_for_status()
    data = r.json()

    # 2) normalizza root & structure
    root = data.get("data", data)
    structure = data.get("structure") or root.get("structure") or {}
    datasets = root.get("dataSets") or data.get("dataSets")
    if not datasets:
        raise ValueError(f"Nessun dataset trovato. Top keys: {list(data.keys())}")

    observations = datasets[0].get("observations")
    if observations is None:
        # alcune risposte annidano le obs dentro 'series'
        series = datasets[0].get("series")
        if series:
            observations = next(iter(series.values())).get("observations")
    if observations is None:
        raise ValueError("Nessuna 'observations' trovata nella risposta SDMX.")

    # 3) mappa codelist per etichette
    codelist_maps = _build_codelist_maps(structure)

    # 4) dimensioni serie (per decodificare la chiave) + dimensioni osservazione (tempo)
    dims_series = (structure.get("dimensions") or {}).get("series", [])
    dims_obs = (structure.get("dimensions") or {}).get("observation", [])
    # il tempo è normalmente la prima dimensione osservazione
    time_values = (dims_obs[0].get("values") if dims_obs else []) or []

    # 5) decodifica chiave serie in meta (code + label) usando le codelist
    #    chiave completa = key (es. "M.IT.N.A.A20.A.1.U2.2240.Z01.E")
    key_parts = key.split(".")
    series_meta = {}
    for i, dim in enumerate(dims_series):
        dim_id = dim.get("id")
        code = key_parts[i] if i < len(key_parts) else None
        # trova codelist collegata alla dimensione
        rep = (dim.get("localRepresentation") or {}).get("enumeration") or {}
        cl_ref = (rep.get("ref") or {}).get("id")
        label = _label_from_codelist(codelist_maps, cl_ref, code) if (cl_ref and code) else code
        if dim_id:
            series_meta[dim_id] = {"code": code, "label": label, "codelist": cl_ref}

    # 6) attributi osservazione: codici -> label
    obs_attrs = (structure.get("attributes") or {}).get("observation", [])
    attr_defs = []
    for attr in obs_attrs:
        a_id = attr.get("id")
        rep = (attr.get("localRepresentation") or {}).get("enumeration") or {}
        cl_ref = (rep.get("ref") or {}).get("id")
        attr_defs.append((a_id, cl_ref))

    # 7) costruisci DataFrame
    rows = []
    for k, v in observations.items():
        # k può essere "0" oppure "0:1" ... primo indice = tempo
        idx = [int(x) for x in k.split(":")]
        t_idx = idx[0]
        date = time_values[t_idx]["id"] if t_idx < len(time_values) else None
        value = v[0] if len(v) > 0 else None

        row = {"TIME_PERIOD": date, "VALUE": value}
        # attributi (posizioni 1..n in v)
        for pos, (a_id, cl_ref) in enumerate(attr_defs, start=1):
            code = v[pos] if pos < len(v) else None
            if code is not None:
                # se l'attributo è indicizzato, mappa su codelist
                if isinstance(code, int):
                    # molti attributi sono indicizzati sulla loro values list
                    # recupera la label dal codelist
                    row[a_id] = code
                    row[f"{a_id}_LABEL"] = _label_from_codelist(codelist_maps, cl_ref,(structure.get("attributes", {}).get("observation", [])[pos-1].get("values", [{}])[code].get("id", code)) if cl_ref else code)
                else:
                    row[a_id] = code
        rows.append(row)

    df = pd.DataFrame(rows).sort_values("TIME_PERIOD").reset_index(drop=True)
    return df, series_meta


In [ ]:
# --------------------------
# Example:
# --------------------------
flow = "BSI"
key  = "M.IT.N.A.A20.A.1.U2.2240.Z01.E"
df, meta = ecb_series_with_metadata(flow, key)
df_meta =pd.DataFrame(meta).T.reset_index().rename(columns={"index": "dimension_id"})
display(df_meta)
display(df.head())

,dimension_id,code,label,codelist
0,FREQ,M,M,None
1,REF_AREA,IT,IT,None
2,ADJUSTMENT,N,N,None
3,BS_REP_SECTOR,A,A,None
4,BS_ITEM,A20,A20,None
5,MATURITY_ORIG,A,A,None
6,DATA_TYPE,1,1,None
7,COUNT_AREA,U2,U2,None
8,BS_COUNT_SECTOR,2240,2240,None
9,CURRENCY_TRANS,Z01,Z01,None


,TIME_PERIOD,VALUE,OBS_STATUS,OBS_STATUS_LABEL,OBS_CONF,OBS_CONF_LABEL
0,2003-01,550083,0,0,0,0
1,2003-02,552340,0,0,0,0
2,2003-03,551633,0,0,0,0
3,2003-04,552928,0,0,0,0
4,2003-05,552368,0,0,0,0


In [ ]:
class DataDownloader:
    def __init__(self, start_date, end_date=None, data_dir='data'):
        self.start_date = start_date
        self.end_date = end_date if end_date else datetime.now().strftime("%Y-%m-%d")
        self.data_dir = data_dir

        os.makedirs(self.data_dir, exist_ok=True)
        print(f"📅 Data collection period: {self.start_date} to {self.end_date}")
        print(f"📂 Data will be saved in: {self.data_dir}")

    def save_data(self, data, filename, overwrite=True):
        """Save data to CSV file if it's a pandas DataFrame"""
        if isinstance(data, pd.DataFrame):
            filepath = os.path.join(self.data_dir, filename)
            if os.path.exists(filepath) and not overwrite:
                print(f"⏩ Skipped saving {filename} (already exists).")
                return
            else:
                data.to_csv(filepath, index=True)
                print(f"✅ Saved: {filename}")
        else:
            print(f"⚠️ Data for {filename} is not a DataFrame and will not be saved.")
            return

    def load(self, download_func, filename, force_download=True):
        """Load data from file or download if not exists"""
        filepath = os.path.join(self.data_dir, filename)
        if force_download:
            data = download_func()
            if not isinstance(data, pd.DataFrame):
                print(f"❌ Downloaded data for {filename} is not a valid DataFrame")
                return None
            self.save_data(data, filename)
        else:
            if os.path.exists(filepath):
                print(f"📁 Loading existing: {filename}")
                data = pd.read_csv(filepath, index_col=0, parse_dates=True)
                if not isinstance(data, pd.DataFrame):
                    print(f"❌ Loaded file {filename} is not a valid DataFrame")
                    return None
            else:
                print(f"⚠️ File {filename} does not exist, attempting download...")
                data = download_func()
                if not isinstance(data, pd.DataFrame):
                    print(f"❌ Downloaded data for {filename} is not a valid DataFrame")
                    return None
                self.save_data(data, filename)

        return data


In [ ]:
def load_from_github(url:str, sheet:str):
    """
    Downloads an Excel file from a GitHub raw URL and loads it into a pandas DataFrame.
    Args:
        url (str): The raw GitHub URL to the .xlsx file.
    Returns:
        pandas.DataFrame: The DataFrame containing the data from the Excel file,
                          or None if an error occurred.
    """
    try:
        print(f"📥 Downloading series list from: {url}")
        response = requests.get(url)
        response.raise_for_status() # Raise an exception for bad status codes
        df = pd.read_excel(io.BytesIO(response.content),sheet_name=sheet)
        print("✅ Series list downloaded and loaded successfully.") # Read the Excel file into a pandas DataFrame
        return df

    except requests.exceptions.RequestException as e:
        print(f"❌ Error downloading the file: {e}")
        return None
    except Exception as e:
        print(f"❌ An error occurred: {e}")
        return None

def load_tickers(url:str, sheet:str, load_from_git: bool = True) -> pd.DataFrame:
    """
    Load the dataset either from GitHub or through a file dialog.
    Args:
        load_from_git (bool): If True, load data from GitHub.
        If False, open a file dialog.
    Returns:
        pd.DataFrame: The loaded dataset.
    """
    if load_from_git:
        return pd.read_excel(url,sheet_name=sheet)
    else:
        #Load data through file dialog
        root = tk.Tk()
        root.withdraw()
        file_path = filedialog.askopenfilename(filetypes=[("xlsx files", "*.xlsx")])
        return pd.read_excel(file_path, sheet)


In [ ]:
github_excel_url = "https://github.com/GMISSAGLIA/GM_PyLab/raw/5093cc6c13d7ccb7eb6872c3422fe0b240606081/T_DECO_TICKERS.xlsx"
DF_TICKERS_ECB = load_from_github(github_excel_url, 'T_DECO_ECB')
DF_TICKERS_ECB = DF_TICKERS_ECB.drop_duplicates(subset=["FULL_CODE"], keep="first")

📥 Downloading series list from: https://github.com/GMISSAGLIA/GM_PyLab/raw/5093cc6c13d7ccb7eb6872c3422fe0b240606081/T_DECO_TICKERS.xlsx
✅ Series list downloaded and loaded successfully.


In [ ]:
def create_base_dataframe(dt_start, dt_end, freq='D'):
    """
    Create base DataFrame with complete date range as index
    Parameters:
    -----------
    dt_start : str or datetime
        Start date (e.g., '2023-01-01' or datetime(2023, 1, 1))
    dt_end : str or datetime
        End date (e.g., '2023-12-31' or datetime(2023, 12, 31))
    freq : str, default 'D'
        Frequency ('D'=daily, 'B'=business days, 'W'=weekly, 'M'=monthly)
    Returns:
    --------
    pd.DataFrame with DatetimeIndex containing all dates in range
    """
    date_range = pd.date_range(start=dt_start, end=dt_end, freq=freq)
    base_df = pd.DataFrame(index=date_range)
    base_df.index.name = 'date'
    return base_df

#example
#dt_start = '1998-12-31'
#dt_end = pd.Timestamp.now().strftime('%Y-%m-%d')
#df_base = create_base_dataframe(dt_start, dt_end, freq='D')
#df_base.head()
df_base = create_base_dataframe(DT_START, DT_END, freq='Q')
df_base.head()

""
date
2001-12-31
2002-03-31
2002-06-30
2002-09-30
2002-12-31


In [ ]:
def filter_columns(df, maxmissing):
    """Filters columns of a DataFrame, keeping only those with less than maxmissing non-null values.
    Args:
        df (pd.DataFrame): The input DataFrame.
        maxmissing (int): The maximum number of missing values allowed for a column to be kept.
    Returns:
        pd.DataFrame: The DataFrame with filtered columns.
    """
    missing = df.isnull().sum()
    columns_to_keep = missing[missing < maxmissing].index.tolist()
    df_filtered = df[columns_to_keep]
    return df_filtered # Return the filtered DataFrame

def transform_ecb(df:pd.DataFrame, name:str, dropcols:bool=True, dropna:bool = True, FREQ: str = "Q", MAXMISSING:int = 1):
    if dropna:
        df.dropna(subset=['OBS_VALUE'], inplace=True)
    df['PROVIDER'] = 'ECB'
    df['NAME'] = name
    df = df.rename(columns={'TIME_PERIOD': 'date', 'OBS_VALUE': 'value'})
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    # Ensure 'date' is the index and drop rows with NaT dates
    df = df.dropna(subset=['date']).set_index('date')
    # Select relevant columns before resampling
    if dropcols:
        cols_to_keep = ['PROVIDER', 'KEY', 'NAME', 'value']
        # Only include 'FREQ' if it exists and is relevant after filtering/dropping
        if 'FREQ' in df.columns:
             cols_to_keep.append('FREQ')
        df = df[cols_to_keep].copy() # Use copy to avoid SettingWithCopyWarning
    if dropna:
        df = filter_columns(df, MAXMISSING)
    # Resample to quarterly if required and the index is a DatetimeIndex
    if FREQ == "QE" and isinstance(df.index, pd.DatetimeIndex):
        df = df.resample('Q').last()

    return df # Reset index to bring date back as a column


def Get_ECB(name:str,flow_type:str, flow_id:str,
            dt_start:str ='1920-01-01', dt_end:str = None,
            BASE_URL:str ="https://data-api.ecb.europa.eu/service/data", dropna: bool = False, timeout: int = 300, dropcols:bool=True, FREQ:str="QE"):
    if dt_end is None:
     dt_end = pd.Timestamp.now().strftime('%Y-%m-%d')
    url = f"{BASE_URL}/{flow_type}/{flow_id}"
    params = {"startPeriod": dt_start, "endPeriod":dt_end,"format":"csvdata"}
    r = requests.get(url, params=params, timeout=timeout); r.raise_for_status()
    df = pd.read_csv(io.StringIO(r.text))

    df = transform_ecb(df, name, dropna=dropna, dropcols=dropcols, FREQ=FREQ)
    return df

def Download_ECB_0(ECB_Tickers: dict, dt_start:str ='1920-01-01', dt_end:str = None, dropna:bool = False,dropcols:bool = True, FREQ = "QE") :
  if dt_end is None:
     dt_end = pd.Timestamp.now().strftime('%Y-%m-%d')
  dfs = {name: Get_ECB(name,code.split('.', 1)[0], code.split('.', 1)[1], dt_start, dt_end, dropna=dropna,dropcols=dropcols, FREQ=FREQ) for name, code in ECB_Tickers.items()}
  return dfs

def Download_ECB(ECB_Tickers: pd.DataFrame, dt_start:str ='1920-01-01', dt_end:str = None, dropna:bool = False,dropcols:bool = True, FREQ = "QE") :
  if dt_end is None:
     dt_end = pd.Timestamp.now().strftime('%Y-%m-%d')
  dfs = {}
  for NAME, DATAFLOW, CODE in ECB_Tickers[["NAME", "DATAFLOW", "CODE"]].itertuples(index=False, name=None):
      try:
          dfs[NAME] = Get_ECB(NAME, DATAFLOW, CODE, dt_start, dt_end, dropna=dropna, dropcols=dropcols, FREQ=FREQ)
      except Exception as e:
          print(f"Error downloading data for ticker {NAME} ({DATAFLOW}/{CODE}): {e}")
          dfs[NAME] = None # Store None for failed downloads
  return dfs


def concat(dfs:dict, dt_start:str, dt_end:str, FREQ:str = "Q", mode:str = 'H' ):
  if mode == 'H':
    df_all = create_base_dataframe(dt_start, dt_end, freq=FREQ)
    for key, df in dfs.items():

      if df is None or not isinstance(df, pd.DataFrame):
        print(f"⚠️ Skipping '{key}' — not a valid DataFrame.")
        continue

      if 'value' not in df.columns:
        print(f"⚠️ Skipping '{key}' — missing 'value' column.")
        continue

      df_temp = df[['value']].copy()
      df_temp = df_temp.rename(columns={'value':key})
      df_all = df_all.join(df_temp, how='left')
  else:
    df_all = pd.concat(dfs.values(), ignore_index=True)
    df_all.dropna(subset=['date'], inplace=True)
  return df_all

def INFO(dfs:dict):
    print("Type of dfs:", type(dfs))
    if isinstance(dfs, dict):
        print("Keys in dfs:", dfs.keys())
        for key, value in dfs.items():
          print(f"--- Content of '{key}' ---")
          print("Type:", type(value))
          if isinstance(value, pd.DataFrame):
            print("Shape:", value.shape)
            value.info()
            #display(value.head())
          else:
            print("Value:", value)
            exit
    else:
        print("dfs is not a dictionary.")

def convert_time_period_0(tp_series: pd.Series, df:pd.DataFrame) -> pd.Series:
    t = tp_series.astype(str)
    is_q = t.str.contains("Q", na=False)
    is_m = t.str.match(r"\d{4}-\d{2}$", na=False)
    date = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
    if is_q.any():
        date.loc[is_q] = pd.PeriodIndex(t[is_q], freq="Q").to_timestamp(how="end")
    if is_m.any():
        date.loc[is_m] = pd.PeriodIndex(t[is_m], freq="M").to_timestamp(how="end")
    if (~(is_q | is_m)).any():
        date.loc[~(is_q | is_m)] = pd.to_datetime(t[~(is_q | is_m)] + "-12-31", errors="coerce")
    return date

def convert_time_period(tp_series: pd.Series) -> pd.Series:

    #Because EUROSTAT uses a non stadard format fro TIME-PERIOD you
    #have to converts Eurostat TIME_PERIOD strings to pandas datetime objects.
    #Automatically detects frequency: Quarterly, Monthly, or Annual.

    if tp_series.str.contains(r'Q\d$', na=False).any():
        # Quarterly format (e.g., '2024Q1')
        return tp_series.apply(lambda x: pd.Period(x, freq='Q').to_timestamp(how='end'))

    elif tp_series.str.contains(r'M\d{2}$', na=False).any():
        # Monthly format (e.g., '2024M01')
        return tp_series.apply(lambda x: pd.Period(x, freq='M').to_timestamp(how='end'))

    elif tp_series.str.fullmatch(r'\d{4}', na=False).any():
        # Annual format (e.g., '2024')
        return tp_series.apply(lambda x: pd.Period(x, freq='A').to_timestamp(how='end'))
    else:
        # Fallback: let pandas try its best
        return pd.to_datetime(tp_series, errors='coerce')

In [ ]:
# Retrive and concat ECB data
dfs = Download_ECB(DF_TICKERS_ECB,dropna=True)
df_all_ecb = concat(dfs,DT_START, DT_END)
df_all_ecb.head()

,LabProdHours_Value_EU27,LabProdHours_Index_EU27,LabProdHours_Index_IT,LabProdHours_Value_IT,LabProdPersons_Value_EU27,LabProdPersons_Index_EU27,LabProdPersons_Index_IT,LabProdPersons_Value_IT,UnitLabourCostHours_Value_EU27,UnitLabourCostHours_Index_IT,...,RETAIL_TURNOVER_INDEX_EU20,AUTOMOTIVE_TURNOVER_INDEX_EU20,CURRENT_ACCOUNT_VALUE_IT,CAPITAL_ACCOUNT_VALUE_IT,FINANCIAL_ACCOUNTS_VALUE_IT,RESERVES_VALUE_IT,EMPLOYEES_COMPENSATION_VALUE_EU27,EMPLOYEES_COMPENSATION_VALUE_IT,EMPLOYEES_COMPENSATION_INDEX_EU27,EMPLOYEES_COMPENSATION_INDEX_ITT
date,,,,,,,,,,,,,,,,,,,,,
2001-12-31,35.128942,82.010418,95.244232,42.431100,14871.687032,90.792807,114.438413,19222.891000,0.450070,79.853859,...,106.3,122.4,2450,519,6998,52436,NaN,NaN,67.468062,81.095618
2002-03-31,35.521403,82.926640,95.234304,42.426677,14973.383087,91.413670,113.572455,19077.430891,0.448235,67.834444,...,82.8,120.5,-1205,505,103,54296,NaN,NaN,67.779855,81.077061
2002-06-30,35.530802,82.948584,95.096412,42.365246,15021.866522,91.709665,114.169917,19177.790143,0.451205,72.275987,...,82.4,122.2,-4253,-562,-1809,49651,NaN,NaN,68.208150,81.821261
2002-09-30,35.797438,83.571059,95.165699,42.396114,15084.673485,92.093106,113.508409,19066.672850,0.453325,68.950921,...,83.4,121.5,3074,-48,4817,53017,NaN,NaN,68.851335,82.223211
2002-12-31,35.916797,83.849710,95.627437,42.601817,15111.767296,92.258516,113.819582,19118.942342,0.454295,82.301858,...,105.9,122.6,-4340,60,-3459,53039,NaN,NaN,69.251400,82.745717


In [ ]:
df_all_ecb

,LabProdHours_Value_EU27,LabProdHours_Index_EU27,LabProdHours_Index_IT,LabProdHours_Value_IT,LabProdPersons_Value_EU27,LabProdPersons_Index_EU27,LabProdPersons_Index_IT,LabProdPersons_Value_IT,UnitLabourCostHours_Value_EU27,UnitLabourCostHours_Index_IT,...,RETAIL_TURNOVER_INDEX_EU20,AUTOMOTIVE_TURNOVER_INDEX_EU20,CURRENT_ACCOUNT_VALUE_IT,CAPITAL_ACCOUNT_VALUE_IT,FINANCIAL_ACCOUNTS_VALUE_IT,RESERVES_VALUE_IT,EMPLOYEES_COMPENSATION_VALUE_EU27,EMPLOYEES_COMPENSATION_VALUE_IT,EMPLOYEES_COMPENSATION_INDEX_EU27,EMPLOYEES_COMPENSATION_INDEX_ITT
date,,,,,,,,,,,,,,,,,,,,,
2001-12-31,35.128942,82.010418,95.244232,42.431100,14871.687032,90.792807,114.438413,19222.891000,0.450070,79.853859,...,106.3,122.4,2450,519,6998,52436,NaN,NaN,67.468062,81.095618
2002-03-31,35.521403,82.926640,95.234304,42.426677,14973.383087,91.413670,113.572455,19077.430891,0.448235,67.834444,...,82.8,120.5,-1205,505,103,54296,NaN,NaN,67.779855,81.077061
2002-06-30,35.530802,82.948584,95.096412,42.365246,15021.866522,91.709665,114.169917,19177.790143,0.451205,72.275987,...,82.4,122.2,-4253,-562,-1809,49651,NaN,NaN,68.208150,81.821261
2002-09-30,35.797438,83.571059,95.165699,42.396114,15084.673485,92.093106,113.508409,19066.672850,0.453325,68.950921,...,83.4,121.5,3074,-48,4817,53017,NaN,NaN,68.851335,82.223211
2002-12-31,35.916797,83.849710,95.627437,42.601817,15111.767296,92.258516,113.819582,19118.942342,0.454295,82.301858,...,105.9,122.6,-4340,60,-3459,53039,NaN,NaN,69.251400,82.745717
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-31,42.862469,100.064758,95.972058,42.755345,17223.191752,105.148927,109.049248,18317.641370,0.675153,116.418959,...,116.2,103.3,5158,7682,17694,224084,NaN,NaN,118.986644,115.170558
2024-03-31,42.862676,100.065241,95.786647,42.672745,17240.611159,105.255274,109.028124,18314.093087,0.683484,102.069857,...,98.4,102.0,315,-477,2243,238866,NaN,NaN,120.559923,116.451667
2024-06-30,42.955080,100.280962,96.290166,42.897061,17267.811051,105.421331,108.862425,18286.259542,0.690118,110.391307,...,99.8,106.3,6239,-1042,-126,251191,NaN,NaN,122.006078,117.499254


# **Using the ecbdata library:**

ecbdata is a wrapper to fetch a series when you already know its full key (e.g. ICP.M.U2.Y.XEF000.3.INX). It doesn’t expose a built-in “list all tickers/series” function.
That said, you can list what’s available with the ECB SDMX API (or pandaSDMX) and then feed any key you find into ecbdata.get_series().

In [ ]:
"""
The ecbdata.get_series function is designed to retrieve data for a full series key
It returns a DataFrame with TIME_PERIOD, OBS_VALUE, plus  some metadata columns.
The detail parameter is supported:
detail="full" (default) returns data + attributes (metadata)
detail="dataonly" returns only the data (no additional metadata)
detail="serieskeysonly" returns only the list of series keys / “tickers”, without the observations (i.e. without time-series values)
So the typical call to list tickers in a dataset is:
"""
from ecbdata import ecbdata

data_key = 'ICP.M.U2.N.000000.4.ANR'
# This retrieves all series keys in the dataset (matching filters) but not the data
df_keys = ecbdata.get_series(data_key, detail="serieskeysonly",start='2024-01', end='2024-03')
df_full= ecbdata.get_series(data_key, detail="full",start='2024-01', end='2024-03')
df_dataonly = ecbdata.get_series(data_key, detail="dataonly",start='2024-01', end='2024-03')
df_full

,KEY,FREQ,REF_AREA,ADJUSTMENT,ICP_ITEM,STS_INSTITUTION,ICP_SUFFIX,TIME_PERIOD,OBS_VALUE,OBS_STATUS,...,PUBL_PUBLIC,UNIT_INDEX_BASE,COMPILATION,COVERAGE,DECIMALS,SOURCE_AGENCY,TITLE,TITLE_COMPL,UNIT,UNIT_MULT
0,ICP.M.U2.N.000000.4.ANR,M,U2,N,0,4,ANR,2024-01,2.8,A,...,NaN,NaN,NaN,NaN,1,NaN,HICP - Overall index,Euro area (changing composition) - HICP - Over...,PCCH,0
1,ICP.M.U2.N.000000.4.ANR,M,U2,N,0,4,ANR,2024-02,2.6,A,...,NaN,NaN,NaN,NaN,1,NaN,HICP - Overall index,Euro area (changing composition) - HICP - Over...,PCCH,0
2,ICP.M.U2.N.000000.4.ANR,M,U2,N,0,4,ANR,2024-03,2.4,A,...,NaN,NaN,NaN,NaN,1,NaN,HICP - Overall index,Euro area (changing composition) - HICP - Over...,PCCH,0


In [ ]:
# We create a wrapper around ecbdata.get_series to make downloads simpler
def download_ecb_data(data_key: str = "FM.B.U2.EUR.4F.KR.MRR_FR.LEV", detail: str = 'full', verbose:bool= False):
    """
    Download one ECB time series via the ecbdata library.

    Parameters
    ----------
    data_key : str
        The full ECB series key that identifies a single time series.
        Example: "EXR.D.USD.EUR.SP00.A" = daily USD/EUR spot rate.
    detail : str
        Level of detail to request from the ECB API.
        Options:
          - "full" : values + attributes + metadata
          - "dataonly" : just time/value data
          - "serieskeysonly" : only list series keys, no observations
          - "nodata" : metadata only, no observations

    Returns
    -------
    pd.DataFrame or None
        A DataFrame with the requested series, or None if download failed.
    """
    if verbose == True:
       print("\n=== ECB DATA DOWNLOAD ===\n")
    try:
        # Step 1: Call ecbdata.get_series with the series key
        if verbose == True:
           print("1. Downloading ECB data...")
        df = ecbdata.get_series(data_key, detail=detail)

        # Step 2: Print summary info
        if verbose == True:
          print(f"Data Frame shape: {df.shape}")
        print(df)

        # Step 3: Return the DataFrame to the caller
        return df

    except ImportError:
        # Special case: library not installed at all
        print("ecbdata library not installed. Run: pip install ecbdata")
        return None

    except Exception as e:
        # Catch any other runtime errors (invalid key, network error, etc.)
        print(f"Error downloading ECB data: {e}")
        return None


In [ ]:
# Yield curve: Euro area, 27Y8M maturity, government bonds
data_key = "YC.B.U2.EUR.4F.G_N_C.SV_C_YM.SR_27Y8M"
download_ecb_data(data_key)

# Monetary Financial Institutions interest rates
data_key = "FM.B.U2.EUR.4F.KR.MRR_FR.LEV"
download_ecb_data(data_key)

# USD/EUR daily spot rate
data_key = "EXR.D.USD.EUR.SP00.A"
download_ecb_data(data_key)

# Balance Sheet Items - example key (Monetary aggregates)
data_key = "BSI.M.U2.Y.V.M30.X.I.U2.2300.Z01.E"
download_ecb_data(data_key)

# Consumer price index (HICP)
data_key = "ICP.M.U2.N.000000.4.ANR"
download_ecb_data(data_key)


                                        KEY FREQ REF_AREA CURRENCY  \
0     YC.B.U2.EUR.4F.G_N_C.SV_C_YM.SR_27Y8M    B       U2      EUR   
1     YC.B.U2.EUR.4F.G_N_C.SV_C_YM.SR_27Y8M    B       U2      EUR   
2     YC.B.U2.EUR.4F.G_N_C.SV_C_YM.SR_27Y8M    B       U2      EUR   
3     YC.B.U2.EUR.4F.G_N_C.SV_C_YM.SR_27Y8M    B       U2      EUR   
4     YC.B.U2.EUR.4F.G_N_C.SV_C_YM.SR_27Y8M    B       U2      EUR   
...                                     ...  ...      ...      ...   
5378  YC.B.U2.EUR.4F.G_N_C.SV_C_YM.SR_27Y8M    B       U2      EUR   
5379  YC.B.U2.EUR.4F.G_N_C.SV_C_YM.SR_27Y8M    B       U2      EUR   
5380  YC.B.U2.EUR.4F.G_N_C.SV_C_YM.SR_27Y8M    B       U2      EUR   
5381  YC.B.U2.EUR.4F.G_N_C.SV_C_YM.SR_27Y8M    B       U2      EUR   
5382  YC.B.U2.EUR.4F.G_N_C.SV_C_YM.SR_27Y8M    B       U2      EUR   

     PROVIDER_FM INSTRUMENT_FM PROVIDER_FM_ID DATA_TYPE_FM TIME_PERIOD  \
0             4F         G_N_C        SV_C_YM     SR_27Y8M  2004-09-06   
1          

,KEY,FREQ,REF_AREA,ADJUSTMENT,ICP_ITEM,STS_INSTITUTION,ICP_SUFFIX,TIME_PERIOD,OBS_VALUE,OBS_STATUS,...,PUBL_PUBLIC,UNIT_INDEX_BASE,COMPILATION,COVERAGE,DECIMALS,SOURCE_AGENCY,TITLE,TITLE_COMPL,UNIT,UNIT_MULT
0,ICP.M.U2.N.000000.4.ANR,M,U2,N,0,4,ANR,1997-01,2.0,A,...,NaN,NaN,NaN,NaN,1,NaN,HICP - Overall index,Euro area (changing composition) - HICP - Over...,PCCH,0
1,ICP.M.U2.N.000000.4.ANR,M,U2,N,0,4,ANR,1997-02,1.8,A,...,NaN,NaN,NaN,NaN,1,NaN,HICP - Overall index,Euro area (changing composition) - HICP - Over...,PCCH,0
2,ICP.M.U2.N.000000.4.ANR,M,U2,N,0,4,ANR,1997-03,1.6,A,...,NaN,NaN,NaN,NaN,1,NaN,HICP - Overall index,Euro area (changing composition) - HICP - Over...,PCCH,0
3,ICP.M.U2.N.000000.4.ANR,M,U2,N,0,4,ANR,1997-04,1.3,A,...,NaN,NaN,NaN,NaN,1,NaN,HICP - Overall index,Euro area (changing composition) - HICP - Over...,PCCH,0
4,ICP.M.U2.N.000000.4.ANR,M,U2,N,0,4,ANR,1997-05,1.4,A,...,NaN,NaN,NaN,NaN,1,NaN,HICP - Overall index,Euro area (changing composition) - HICP - Over...,PCCH,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
339,ICP.M.U2.N.000000.4.ANR,M,U2,N,0,4,ANR,2025-04,2.2,A,...,NaN,NaN,NaN,NaN,1,NaN,HICP - Overall index,Euro area (changing composition) - HICP - Over...,PCCH,0
340,ICP.M.U2.N.000000.4.ANR,M,U2,N,0,4,ANR,2025-05,1.9,A,...,NaN,NaN,NaN,NaN,1,NaN,HICP - Overall index,Euro area (changing composition) - HICP - Over...,PCCH,0
341,ICP.M.U2.N.000000.4.ANR,M,U2,N,0,4,ANR,2025-06,2.0,E,...,NaN,NaN,NaN,NaN,1,NaN,HICP - Overall index,Euro area (changing composition) - HICP - Over...,PCCH,0
342,ICP.M.U2.N.000000.4.ANR,M,U2,N,0,4,ANR,2025-07,2.0,A,...,NaN,NaN,NaN,NaN,1,NaN,HICP - Overall index,Euro area (changing composition) - HICP - Over...,PCCH,0


#=========================================
# ECB DATA using the 'pandaSDMX' library
# ========================================

In [ ]:
# ECB example: daily USD/EUR spot rate since 2024-01-01

import sdmx  # note: this is the modern 'sdmx1' library, not 'pandasdmx'

# 1. Create a client for the ECB's SDMX REST API
req = sdmx.Client("ECB")
# This sets up a connection to the ECB Data Portal,
# so you can request datasets (flows) like EXR (exchange rates), YC (yield curve), etc.

# 2. Request data from the "EXR" dataset (exchange rates)
msg = req.data(
    "EXR",                        # flow ID: exchange rates
    key="D.USD.EUR.SP00.A",       # series key: dimensions specify frequency, currencies, type
    params={"startPeriod": "2024-01-01"}  # request data starting from Jan 1, 2024
)

# Breakdown of the series key "D.USD.EUR.SP00.A":
#   D     = Daily frequency
#   USD   = Currency being quoted
#   EUR   = Base currency
#   SP00  = Spot rate, standard series
#   A     = Observation type (average or raw, depending on flow structure)

# 3. Convert SDMX message object to a pandas Series
ser = sdmx.to_pandas(msg).rename("USD/EUR")
#   - sdmx.to_pandas() parses the SDMX message into a pandas object
#   - For a single series, you get a pandas Series with a PeriodIndex/DatetimeIndex
#   - .rename("USD/EUR") gives the series a friendly column/series name

# 4. Display the time series
ser
# The output is a pandas Series indexed by date (daily),
# with values equal to the USD/EUR exchange rate.


FREQ  CURRENCY  CURRENCY_DENOM  EXR_TYPE  EXR_SUFFIX  TIME_PERIOD
D     USD       EUR             SP00      A           2024-01-02     1.0956
                                                      2024-01-03     1.0919
                                                      2024-01-04     1.0953
                                                      2024-01-05     1.0921
                                                      2024-01-08     1.0946
                                                                      ...  
                                                      2025-09-22     1.1781
                                                      2025-09-23     1.1793
                                                      2025-09-24     1.1756
                                                      2025-09-25     1.1739
                                                      2025-09-26     1.1672
Name: USD/EUR, Length: 445, dtype: float64

In [ ]:
def download_ecb_with_pandasdmx(flow: str, key: str, verbose: bool, **params):
    """
    Download data from the ECB Data Portal using the pandaSDMX (or sdmx1) library.

    Parameters
    ----------
    flow : str
        The dataset (dataflow) ID, e.g. "EXR" (exchange rates), "YC" (yield curve).
    key : str
        The series key that uniquely identifies a time series inside the dataset,
        e.g. "D.USD.EUR.SP00.A" for the daily USD/EUR spot rate.
    verbose : bool
        If True, print diagnostic messages and a data preview.
    **params : dict
        Extra query parameters to pass to the ECB API, such as:
          - startPeriod="2024-01-01"
          - endPeriod="2024-06-30"
          - detail="dataonly"

    Returns
    -------
    pd.DataFrame or None
        A pandas DataFrame containing the requested series, or None if error.
    """

    if verbose:
        print("\n=== ECB with pandaSDMX ===\n")

    try:
        # 1. Create an SDMX client for the ECB API
        ecb = sdmx.Client('ECB')

        if verbose:
            print("1. Downloading with pandaSDMX...")

        # 2. Make a data request
        response = ecb.data(flow, key=key, params=params)
        # response is an SDMX message object containing the data

        # 3. Convert the first dataset in the response to pandas
        df = sdmx.to_pandas(response.data[0]).reset_index()
        # - sdmx.to_pandas() parses the SDMX data into a Series or DataFrame
        # - reset_index() turns the index (time) into a normal column

        if verbose:
            print(f"data shape: {df.shape}")
            print(df.tail())  # preview the last few rows

        return df

    except ImportError:
        # Special case if the library is missing
        print("pandaSDMX library not installed. Run: pip install pandaSDMX")
        return None

    except Exception as e:
        # Catch-all for other errors (bad key, network error, etc.)
        print(f"Error with ECB pandaSDMX: {e}")
        return None


In [ ]:
# First example: Exchange Rates dataset
"""
  flow = 'EXR' → Selects the Exchange Rates dataset.
  key = 'D.USD.EUR.SP00.A' → This is the series key that identifies the daily USD/EUR spot rate.
  D = Daily frequency
  USD = quoted currency
  EUR = base currency
  SP00 = Spot rate, standard series
  A = observation type (average/raw depending on flow)
  download_ecb_with_pandasdmx(..., False, **params) → Calls the wrapper function without verbose logging, passing in the params dict.
  This downloads the daily USD/EUR spot rate from Jan 2020 onwards.
"""
params = {'startPeriod': '2020-01-01'}
flow = 'EXR'
key = 'D.USD.EUR.SP00.A'
df_usd_eur=download_ecb_with_pandasdmx(flow, key, False, **params)


In [ ]:
# Second example: Monetary Financial Institutions interest rates
"""
flow = 'FM' → Selects the Financial Markets dataset.
key = 'B.U2.EUR.4F.KR.MRR_FR.LEV' → A specific series key describing interest rate data.
B = Business frequency
U2 = Euro area aggregate
EUR = currency
4F / KR etc. = dataset-specific dimension codes (here they correspond to “Monetary Financial Institutions, key rate, Main Refinancing Operations, level”)
The function call downloads the ECB main refinancing rate (MRR_FR) for the euro area, from Jan 2020

"""
flow = 'FM'
key = 'B.U2.EUR.4F.KR.MRR_FR.LEV'
df_RATES = download_ecb_with_pandasdmx(flow, key, False, **params)


In [ ]:
def my_ecb_series():
    """
    ECB series codes for reference

    This function returns a dictionary of commonly used ECB time series codes,
    grouped by category (exchange rates, interest rates, etc.).
    It also prints them nicely to the console for quick reference.
    """

    print("\n=== ECB SERIES CODES ===\n")

    # Dictionary of series grouped into categories
    ecb_series = {
        'Exchange Rates': {
            'EUR/USD': 'EXR.D.USD.EUR.SP00.A',
            'EUR/GBP': 'EXR.D.GBP.EUR.SP00.A',
            'EUR/JPY': 'EXR.D.JPY.EUR.SP00.A',
            'EUR/CHF': 'EXR.D.CHF.EUR.SP00.A'
        },
        'Interest Rates': {
            'Main Refinancing Rate': 'FM.B.U2.EUR.4F.KR.MRR_FR.LEV',
            'Deposit Facility Rate': 'FM.B.U2.EUR.4F.KR.DFR.LEV',
            '10Y Government Bond Yield': 'IRS.M.DE.L.L40.CI.0000.EUR.N.Z'
        },
        'Monetary Aggregates': {
            'M1': 'BSI.M.U2.Y.V.M10.X.I.U2.2300.Z01.E',
            'M3': 'BSI.M.U2.Y.V.M30.X.I.U2.2300.Z01.E'
        },
        'Economic Indicators': {
            'HICP - All Items': 'ICP.M.U2.N.000000.4.ANR',
            'Industrial Production': 'STS.M.I8.Y.PROD.NS0010.4.000'
        }
    }

    # Print out all categories and their series
    for category, series in ecb_series.items():
        print(f"\n{category}:")
        for name, code in series.items():
            print(f"  {name}: {code}")

    return ecb_series


In [ ]:
def save_data_to_files(results, output_dir='data'):
    """
    Save downloaded data to CSV files.

    Parameters
    ----------
    results : dict
        A dictionary where keys are names (strings) and values are pandas DataFrames.
        Example: {"EXR_USD_EUR": df1, "FM_MRR_FR": df2}
    output_dir : str, default 'data'
        The folder where CSV files will be saved. If it does not exist,
        the function will create it.
    """

    import os  # local import (could also be placed at the top of the file)

    # 1. Ensure the output directory exists
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # 2. Loop through all results
    for name, data in results.items():
        # Only save objects that are DataFrames
        if isinstance(data, pd.DataFrame):
            # Construct the full path: e.g. "data/EXR_USD_EUR.csv"
            filename = os.path.join(output_dir, f"{name}.csv")

            # Save DataFrame to CSV
            data.to_csv(filename)

            # Confirm to the user
            print(f"Saved {name} to {filename}")


results = {
    "EXR_USD_EUR": df_usd_eur,
    "FM_MRR_FR": df_RATES,
}
save_data_to_files(results)


Saved EXR_USD_EUR to data/EXR_USD_EUR.csv
Saved FM_MRR_FR to data/FM_MRR_FR.csv
